In [1]:
import json

# Path to your JSON file
file_path = "data/dataset_v1/properties/Arabic.json"

# Read the JSON file
with open(file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

In [3]:
# Function to find entities with labels_count != 9
def find_entities_with_incomplete_labels(data):
    results = {}
    
    for entity_id, entity_data in data.items():
        # Check the entity's own labels count
        if entity_data.get('labels_count', 0) != 9:
            if entity_id not in results:
                results[entity_id] = []
            results[entity_id].append({
                'type': 'entity_label',
                'count': entity_data.get('labels_count', 0)
            })
        
        # Check property values
        for prop_id, prop_data in entity_data.items():
            if prop_id not in ['labels', 'labels_count'] and isinstance(prop_data, dict) and 'values' in prop_data:
                for value in prop_data['values']:
                    if value.get('labels_count', 0) != 9:
                        if entity_id not in results:
                            results[entity_id] = []
                        results[entity_id].append({
                            'property': prop_id,
                            'qid': value.get('qid', 'unknown'),
                            'count': value.get('labels_count', 0)
                        })
    
    return results

In [5]:
incomplete_labels = find_entities_with_incomplete_labels(data)

# Print results
print(f"Found {len(incomplete_labels)} entities with incomplete labels:")
for entity_id, details in incomplete_labels.items():
    print(f"\nEntity: {entity_id}")
    for detail in details:
        if 'type' in detail and detail['type'] == 'entity_label':
            print(f"  - Entity's own labels count: {detail['count']}")
        else:
            print(f"  - Property {detail['property']}, QID: {detail['qid']}, labels count: {detail['count']}")

Found 8035 entities with incomplete labels:

Entity: Q469901
  - Entity's own labels count: 5
  - Property P569, QID: 1976-10-01T00:00:00Z, labels count: 0

Entity: Q462287
  - Entity's own labels count: 6
  - Property P569, QID: 1926-11-11T00:00:00Z, labels count: 0
  - Property P569, QID: 1931-11-12T00:00:00Z, labels count: 0

Entity: Q463514
  - Entity's own labels count: 8
  - Property P569, QID: 1979-01-20T00:00:00Z, labels count: 0

Entity: Q462256
  - Entity's own labels count: 8
  - Property P569, QID: 1973-08-15T00:00:00Z, labels count: 0

Entity: Q1036918
  - Entity's own labels count: 5
  - Property P569, QID: 1963-09-13T00:00:00Z, labels count: 0

Entity: Q1103528
  - Entity's own labels count: 5
  - Property P569, QID: 1980-11-15T00:00:00Z, labels count: 0

Entity: Q1033841
  - Entity's own labels count: 6
  - Property P569, QID: 1900-09-02T00:00:00Z, labels count: 0

Entity: Q1009512
  - Entity's own labels count: 4
  - Property P569, QID: 1961-12-23T00:00:00Z, labels cou

In [8]:
import json

# Path to your JSON file
file_path = "data/dataset_v1/properties/Arabic.json"

# Expected languages
label_languages = ["en", "de", "fr", "ru", "hi", "zh", "it", "pl", "ar"]

# Function to find missing languages in labels
def find_missing_languages(data):
    results = {}
    
    for entity_id, entity_data in data.items():
        entity_missing = {}
        
        # Check the entity's own labels
        if 'labels' in entity_data:
            present_langs = set(entity_data['labels'].keys())
            missing_langs = set(label_languages) - present_langs
            if missing_langs:
                entity_missing['entity_labels'] = list(missing_langs)
        
        # Check property values
        for prop_id, prop_data in entity_data.items():
            if prop_id not in ['labels', 'labels_count', 'P569', 'P570'] and isinstance(prop_data, dict) and 'values' in prop_data:
                for i, value in enumerate(prop_data['values']):
                    if 'labels' in value:
                        present_langs = set(value['labels'].keys())
                        missing_langs = set(label_languages) - present_langs
                        if missing_langs:
                            key = f"{prop_id}_value_{i}"
                            entity_missing[key] = {
                                'qid': value.get('qid', 'unknown'),
                                'missing_languages': list(missing_langs)
                            }
        
        # Add to results if any missing languages were found
        if entity_missing:
            results[entity_id] = entity_missing
    
    return results

# Read the JSON file
with open(file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# Find missing languages
missing_languages = find_missing_languages(data)

# Print results
print(f"Found {len(missing_languages)} entities with missing languages:")
for entity_id, details in missing_languages.items():
    print(f"\nEntity: {entity_id}")
    
    if 'entity_labels' in details:
        print(f"  Entity's own missing languages: {', '.join(details['entity_labels'])}")
    
    for key, value in details.items():
        if key != 'entity_labels':
            prop_id = key.split('_value_')[0]
            print(f"  Property {prop_id}, QID: {value['qid']}")
            print(f"    Missing languages: {', '.join(value['missing_languages'])}")

Found 7978 entities with missing languages:

Entity: Q469901
  Entity's own missing languages: hi, ru, pl, it

Entity: Q462287
  Entity's own missing languages: hi, pl, zh

Entity: Q463514
  Entity's own missing languages: zh

Entity: Q462256
  Entity's own missing languages: hi

Entity: Q1036918
  Entity's own missing languages: hi, it, pl, zh

Entity: Q1103528
  Entity's own missing languages: hi, ru, pl, zh

Entity: Q1033841
  Entity's own missing languages: hi, ru, zh

Entity: Q1009512
  Entity's own missing languages: hi, ru, it, pl, zh

Entity: Q1092010
  Entity's own missing languages: hi, it, pl, zh

Entity: Q1033909
  Entity's own missing languages: hi

Entity: Q1068717
  Entity's own missing languages: hi, ru, it, pl, zh

Entity: Q984486
  Entity's own missing languages: hi
  Property P19, QID: http://www.wikidata.org/entity/Q108394982
    Missing languages: hi, ru, de, fr, it, pl, zh
  Property P27, QID: http://www.wikidata.org/entity/Q146885
    Missing languages: hi, it

E

In [1]:
import json
import requests
from googletrans import Translator
import time

# Path to your JSON file
file_path = "data/dataset_v1/properties/Arabic.json"

# Expected languages
label_languages = ["en", "de", "fr", "ru", "hi", "zh", "it", "pl", "ar"]

# Initialize translator
translator = Translator()

# Function to translate text to multiple languages
def translate_to_languages(text, source_lang, target_langs):
    translations = {}
    
    # If source text is empty, return empty translations
    if not text:
        return {lang: "" for lang in target_langs}
    
    for lang in target_langs:
        try:
            # Skip if target language is the same as source language
            if lang == source_lang:
                translations[lang] = text
                continue
                
            # Add delay to avoid rate limiting
            time.sleep(0.5)
            
            # Translate text
            translation = translator.translate(text, src=source_lang, dest=lang)
            translations[lang] = translation.text
            
            print(f"Translated '{text}' from {source_lang} to {lang}: {translation.text}")
        except Exception as e:
            print(f"Error translating to {lang}: {e}")
            translations[lang] = ""
    
    return translations

# Function to find and complete missing languages
def complete_missing_languages(data):
    updated_count = 0
    
    for entity_id, entity_data in data.items():
        # Complete entity's own labels
        if 'labels' in entity_data:
            present_langs = set(entity_data['labels'].keys())
            missing_langs = set(label_languages) - present_langs
            
            if missing_langs and present_langs:
                # Find a source language for translation (preferably English)
                source_lang = 'en' if 'en' in present_langs else next(iter(present_langs))
                source_text = entity_data['labels'][source_lang]
                
                # Translate to missing languages
                translations = translate_to_languages(source_text, source_lang, missing_langs)
                
                # Update labels
                for lang, translation in translations.items():
                    if translation:
                        entity_data['labels'][lang] = translation
                        updated_count += 1
                
                # Update labels count
                entity_data['labels_count'] = len(entity_data['labels'])
        
        # Complete property values labels
        for prop_id, prop_data in entity_data.items():
            if prop_id not in ['labels', 'labels_count', 'P569', 'P570'] and isinstance(prop_data, dict) and 'values' in prop_data:
                for value in prop_data['values']:
                    if 'labels' in value:
                        present_langs = set(value['labels'].keys())
                        missing_langs = set(label_languages) - present_langs
                        
                        if missing_langs and present_langs:
                            # Find a source language for translation
                            source_lang = 'en' if 'en' in present_langs else next(iter(present_langs))
                            source_text = value['labels'][source_lang]
                            
                            # Translate to missing languages
                            translations = translate_to_languages(source_text, source_lang, missing_langs)
                            
                            # Update labels
                            for lang, translation in translations.items():
                                if translation:
                                    value['labels'][lang] = translation
                                    updated_count += 1
                            
                            # Update labels count
                            value['labels_count'] = len(value['labels'])
    
    return data, updated_count

# Read the JSON file
with open(file_path, 'r', encoding='utf-8') as file:
    data = json.load(file)

# Process a small subset for testing (remove this limitation for full processing)
test_data = {k: data[k] for k in list(data.keys())[:10]}

# Complete missing languages
print("Starting translation process...")
updated_data, updated_count = complete_missing_languages(test_data)

print(f"Updated {updated_count} missing language labels")

# Save the updated data to a new file
output_file = "data/dataset_v1/properties/Arabic_completed.json"
with open(output_file, 'w', encoding='utf-8') as file:
    json.dump(updated_data, file, ensure_ascii=False, indent=2)

print(f"Updated data saved to {output_file}")

# Function to verify completeness
def verify_completeness(data):
    incomplete_entities = 0
    
    for entity_id, entity_data in data.items():
        is_incomplete = False
        
        # Check entity's own labels
        if 'labels' in entity_data:
            if len(entity_data['labels']) < len(label_languages):
                is_incomplete = True
        
        # Check property values
        for prop_id, prop_data in entity_data.items():
            if prop_id not in ['labels', 'labels_count'] and isinstance(prop_data, dict) and 'values' in prop_data:
                for value in prop_data['values']:
                    if 'labels' in value and len(value['labels']) < len(label_languages):
                        is_incomplete = True
        
        if is_incomplete:
            incomplete_entities += 1
    
    return incomplete_entities

# Verify completeness of the updated data
incomplete_count = verify_completeness(updated_data)
print(f"Remaining incomplete entities: {incomplete_count} out of {len(updated_data)}")

ModuleNotFoundError: No module named 'googletrans'